# 실습 2 · 밴딧 문제와 톰슨 샘플링

**인공지능 일반 · 3차시**

지난 시간에 우리는 슬롯머신 5대를 손으로 100번 당겨봤습니다.
오늘은 그 일을 **코드가 하게** 만듭니다.

| 순서 | 내용 |
|---|---|
| 1 | 환경 만들기 — 슬롯머신 5대 |
| 2 | **그리디** — 좋아 보이는 것만 계속 |
| 3 | **ε-그리디** — 가끔 딴짓하기 |
| 4 | 베타 분포 감각 잡기 |
| 5 | **톰슨 샘플링** — 분포에서 제비 뽑기 |
| 6 | 학습 과정 들여다보기 |
| 7 | 세 전략 정면 비교 |

> 셀 실행은 `Shift + Enter`. 위에서부터 **순서대로** 실행하세요.

## ✏️ 이 노트북은 실습용입니다

코드 중간중간 `여기를_채우세요` 로 표시된 **빈칸 5곳**이 있습니다. 주석에 적힌 힌트를 읽고 빈칸을 채운 뒤 셀을 실행하세요.

빈칸을 채우지 않고 실행하면 `NameError: name '여기를_채우세요' is not defined` 에러가 납니다 — **"아직 못 채웠다"는 뜻**이니 당황하지 말고 그 칸을 채우면 됩니다.

| 번호 | 위치 | 무엇을 채우나요 |
|---|---|---|
| ① | 환경 만들기 | `pull()` — 확률적으로 당첨/꽝을 돌려주기 |
| ② | 그리디 | 추정 확률 계산 + 1등 기계 고르기 |
| ③ | ε-그리디 | 탐험 여부 판단 + 무작위로 기계 고르기 |
| ④ | 톰슨 샘플링 | 베타 분포에서 숫자 뽑기 |
| ⑤ | 톰슨 샘플링 | 가장 큰 숫자의 기계 고르기 |

## 0. 준비

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('준비 완료!')

준비 완료!


## 0-1. 앞으로 쓸 무작위 도구 4가지

| 도구 | 하는 일 |
|---|---|
| `random.random()` | 0~1 사이 실수 하나. `< 0.65` 면 **65% 확률로 참** → 당첨/꽝 판정 |
| `random.randrange(K)` | 0~K-1 정수 하나 → **기계 아무거나 하나 고르기** (탐험) |
| `random.seed(n)` | 씨앗 고정 → 같은 씨앗이면 **같은 난수 반복** (재현용) |
| `np.random.beta(a, b)` | **Beta(a,b) 분포에서 숫자 하나** 뽑기 → 톰슨 샘플링의 핵심 |

`np.random.beta(당첨+1, 꽝+1)` 은 "이 기계 확률이 얼마쯤일까"라는 믿음에서 제비를 뽑는 것.
많이 당겨본 기계는 좁은 범위에서, 안 당겨본 기계는 0~1 넓게 뽑힙니다.

아래 셀에서 `SEED` 를 바꿔가며 실행해 보세요.

In [ ]:
SEED = 0          # ← 0, 1, 42 ... 로 바꿔가며 실행해 보세요
names = ['A', 'B', 'C', 'D', 'E']

# 0) 씨앗을 매번 새로(인자 없이) 뽑으면 → 셀을 다시 실행할 때마다 값이 달라진다
random.seed()     # 인자 없음 = OS 난수/시간으로 새 씨앗
print('random.seed() 후 random.random() 3번 (재실행하면 매번 바뀜):')
for t in range(3):
    print('  ', round(random.random(), 3))

# 1) random.seed(고정값) 을 주면 → 항상 같은 숫자가 나온다
random.seed(SEED)
print('\nseed 고정 후 random.random() 5번:')
for t in range(5):
    print('  ', round(random.random(), 3))

# 씨앗을 같은 값으로 다시 고정하면 → 똑같은 숫자가 반복된다
random.seed(SEED)
print('seed 재고정 후 5번 (위와 똑같이 나옴):')
for t in range(5):
    print('  ', round(random.random(), 3))

# random.random() < p  →  p 확률로 True
p = 0.65
random.seed(SEED)
hits = 0
for t in range(1000):
    if random.random() < p:
        hits += 1
print(f'\nrandom.random() < {p} 를 1000번 → 참 {hits}번 (기대 {p * 1000:.0f}번)\n')

# 2) random.randrange(K) : 0~K-1 정수 하나 (기계 번호 뽑기)
random.seed(SEED)
print('randrange(5) 10번:')
for t in range(10):
    print('  ', random.randrange(5))

# 1000번 뽑아서 번호별로 몇 번 나왔는지 세기
random.seed(SEED)
count = [0, 0, 0, 0, 0]
for t in range(1000):
    i = random.randrange(5)
    count[i] = count[i] + 1
print('\nrandrange(5) 1000번 → 번호별 횟수 (기대 200번씩, 조금씩 흔들림):')
for i in range(5):
    print(f'  {i}번 (기계 {names[i]}) : {count[i]}번')

# 3) np.random.beta(a, b) : Beta 분포에서 숫자 하나
np.random.seed(SEED)
print('\nnp.random.beta(당첨+1, 꽝+1) 를 5번씩 뽑기:')
cases = [(1, 1, '1승 1패   → 아직 잘 모름 (넓게 뽑힘)'),
         (8, 2, '8승 2패   → 좋아 보임 (오른쪽에서 뽑힘)'),
         (50, 50, '50승 50패 → 확신 0.5 (좁게 뽑힘)')]
for wins, loses, tag in cases:
    a = wins + 1
    b = loses + 1
    print(f'  Beta({a}, {b})  {tag}')
    for t in range(5):
        print('     ', round(np.random.beta(a, b), 3))


## 1. 환경 만들기 — 슬롯머신 5대

`TRUE_P`는 각 기계가 당첨될 **진짜 확률**입니다.

**중요:** 이 값은 *환경*이 알고 있는 값입니다.
우리가 만들 AI는 이 값을 **절대 들여다보지 않습니다.**
오직 `pull()`을 호출해서 돌아오는 0과 1만 보고 배워야 합니다.

In [ ]:
TRUE_P = [0.40, 0.15, 0.65, 0.25, 0.55]   # AI는 이 값을 모른다!
NAMES  = ['A', 'B', 'C', 'D', 'E']
K      = len(TRUE_P)      # 기계 개수
N      = 1000             # 당길 횟수

def pull(i):
    '''i번 기계를 당긴다 → 1(당첨) 또는 0(꽝)'''
    return 1 if random.random() < TRUE_P[i] else 0

def argmax_random(v):
    '''v = 기계별 숫자가 든 리스트(예: 승률 5개).
       가장 큰 값이 있는 '번호'를 돌려준다. 동점이면 그중 하나를 무작위로.'''
    m = max(v)
    cands = []                       # 최댓값을 가진 번호를 모은다
    for i in range(len(v)):
        if v[i] == m:
            cands.append(i)
    return random.choice(cands)      # 그중 하나를 무작위로

# 기준선 두 개
BEST   = max(TRUE_P)                 # 가장 좋은 기계의 확률
CHANCE = sum(TRUE_P) / K             # 아무렇게나 찍었을 때의 평균

print(f'가장 좋은 기계만 {N}번 당기면   : {BEST * N:.0f}점  ← 천장')
print(f'무작위로 {N}번 당기면          : {CHANCE * N:.0f}점  ← 바닥')

가장 좋은 기계만 1000번 당기면   : 650점  ← 천장
무작위로 1000번 당기면          : 400점  ← 바닥


## 2. 그리디 — 좋아 보이는 것만 계속 당기기

규칙은 단순합니다.

1. 각 기계를 **한 번씩** 당겨본다
2. 그 다음부터는 **추정 확률이 가장 높은 기계**만 계속 당긴다

추정 확률 = `당첨 횟수 / 시도 횟수`

In [ ]:
def greedy(n_pulls=N):
    wins   = [0] * K      # 기계별 당첨 횟수
    trials = [0] * K      # 기계별 시도 횟수
    total  = 0
    history = []          # 매 판 누적 점수 기록

    for t in range(n_pulls):
        if 0 in trials:                  # ① 아직 안 당겨본 기계부터
            i = trials.index(0)
        else:                            # ② 추정 확률 1등
            # ✏️ 실습 ② : 기계별 추정 확률(승률)을 계산해 q에 담고,
            #            그중 가장 큰 값을 가진 기계 번호를 i에 담으세요.
            #            (힌트: 추정 확률 = 당첨 / 시도 · 동점 처리는 argmax_random 사용)
            q = 여기를_채우세요
            i = 여기를_채우세요

        r = pull(i)                      # ③ 당긴다
        trials[i] += 1
        wins[i]   += r
        total     += r
        history.append(total)

    return total, trials, history


random.seed(0)
score, trials, _ = greedy()
print(f'총점 : {score} / {N}')
for i in range(K):
    print(f'  기계 {NAMES[i]} : {trials[i]:4d}번 당김')

### 한 번 돌려본 건 아무 의미가 없습니다

위 결과는 **운이 좋았거나 나빴던 한 판**일 뿐입니다.
씨앗(seed)을 바꿔가며 1,000번 반복해서 **점수의 분포**를 봐야 합니다.

In [ ]:
def repeat(strategy, runs=1000, eps=None):
    '''전략을 runs번 반복 실행하고 점수 목록을 돌려준다.
       eps를 적어 주면 eps_greedy에 그 값을 넘긴다.'''
    scores = []
    for run in range(runs):
        random.seed(run)          # 이 판의 '운'을 고정
        np.random.seed(run)

        if eps is None:                     # greedy, thompson
            total = strategy()[0]
        else:                               # eps_greedy
            total = strategy(eps=eps)[0]

        scores.append(total)      # 총점만 모은다
    return np.array(scores)


g_scores = repeat(greedy, runs=400)

print(f'그리디 {len(g_scores)}회 반복')
print(f'  평균   : {g_scores.mean():.1f}점')
print(f'  표준편차: {g_scores.std():.1f}점   ← 이 값이 크면 "운에 맡긴다"는 뜻')
print(f'  최악   : {g_scores.min()}점')
print(f'  최선   : {g_scores.max()}점')

plt.figure(figsize=(7, 3))
plt.hist(g_scores, bins=30, color='#c8324a', alpha=.75)
plt.axvline(BEST * N,   color='#0e8a52', ls='--', lw=2, label=f'최적 {BEST*N:.0f}점')
plt.axvline(CHANCE * N, color='#8b98ac', ls='--', lw=2, label=f'무작위 {CHANCE*N:.0f}점')
plt.title('그리디의 점수 분포 (400회 반복)')
plt.xlabel('점수'); plt.ylabel('횟수'); plt.legend(); plt.tight_layout(); plt.show()

**무엇이 보이나요?**

봉우리가 **여러 개로 뚝뚝 끊어져** 있습니다. 우연이 아닙니다.
봉우리의 위치를 `TRUE_P`와 비교해 보세요.

`0.15 × 1000 = 150`, `0.25 × 1000 = 250`, `0.40 × 1000 = 400`, `0.55 × 1000 = 550`, `0.65 × 1000 = 650`

**봉우리 하나가 곧 "어느 기계에 갇혔는가"입니다.**
초반 5번에서 어느 기계가 우연히 터졌는지가 1,000번의 운명을 결정한 겁니다.

> **핵심** — 그리디는 성적이 *실력*이 아니라 *초반 운*으로 정해집니다.
> 평균이 아니라 **이 히스토그램의 모양**이 그리디의 진짜 문제입니다.

## 3. ε-그리디 — 동전을 던져서 정한다

- 확률 `1-ε` : 추정 1등 기계를 당긴다 (**활용**)
- 확률 `ε`   : 아무 기계나 당긴다 (**탐험**)

In [ ]:
def eps_greedy(n_pulls=N, eps=0.1):
    wins   = [0] * K
    trials = [0] * K
    total  = 0
    history = []

    for t in range(n_pulls):
        if 0 in trials:                           # 아직 안 당겨본 기계부터
            i = trials.index(0)
        elif 여기를_채우세요:                                  # ✏️ 실습 ③ : 확률 eps로 "탐험"을 선택하는 조건을 쓰세요
            i = 여기를_채우세요                                # ✏️            그리고 무작위로 기계 하나를 고르세요 (힌트: random.randrange(K))
        else:                                     # 앞면 → 활용
            q = []
            for j in range(K):
                q.append(wins[j] / trials[j])
            i = argmax_random(q)

        r = pull(i)
        trials[i] += 1
        wins[i]   += r
        total     += r
        history.append(total)

    return total, trials, history


for eps in [0.0, 0.05, 0.1, 0.3, 1.0]:
    sc = repeat(eps_greedy, runs=150, eps=eps)
    tag = ' ← 그리디와 같음' if eps == 0.0 else (' ← 완전 무작위' if eps == 1.0 else '')
    print(f'ε = {eps:<5}  평균 {sc.mean():6.1f}점   표준편차 {sc.std():5.1f}{tag}')

**생각해 볼 것**

- ε을 키우면 평균은 어떻게 되나요? 표준편차는요?
- 가장 좋은 ε은 몇이었나요? 그걸 **미리** 알 수 있었을까요?

바로 이게 ε-그리디의 약점입니다. **ε을 몇으로 둘지 정답이 없고**,
1,000번을 당겨 답을 다 알아낸 뒤에도 계속 `ε`만큼 낭비합니다.

## 4. 베타 분포 감각 잡기

지금까지 우리는 각 기계를 **숫자 하나**(추정 확률)로만 기억했습니다.
그래서 이런 걸 구분할 수 없었습니다.

- 2번 당겨 1번 성공 → 추정 0.50
- 100번 당겨 50번 성공 → 추정 0.50

둘 다 0.50인데, 확신의 정도는 하늘과 땅 차이죠.
**베타 분포**를 쓰면 이 차이를 그림으로 적을 수 있습니다.

$$\text{Beta}(\alpha, \beta), \qquad \alpha = 당첨 + 1, \quad \beta = 꽝 + 1$$

In [ ]:
from scipy.stats import beta as beta_dist

x = np.linspace(0, 1, 400)
cases = [(0, 0,  '아직 안 당김',      '#8b98ac'),
         (1, 1,  '1승 1패',           '#b06a00'),
         (50, 50,'50승 50패',         '#0e8a52')]

plt.figure(figsize=(7, 3.2))
for w, l, label, color in cases:
    a, b = w + 1, l + 1
    plt.plot(x, beta_dist.pdf(x, a, b), color=color, lw=2.5,
             label=f'{label}  →  Beta({a}, {b})')
plt.axvline(0.5, color='#16233c', ls=':', lw=1.5)
plt.title('당첨 확률이 얼마일지에 대한 "믿음"')
plt.xlabel('당첨 확률'); plt.ylabel('그럴듯함(밀도)')
plt.legend(); plt.tight_layout(); plt.show()

print('세 곡선 모두 평균은 0.5입니다. 다른 건 폭 — 즉 확신의 정도입니다.')
print('아직 안 당긴 기계는 완전히 평평합니다 = "0부터 1까지 뭐든 그럴듯하다"')

## 5. 톰슨 샘플링

세 줄이면 끝납니다.

1. 기계마다 `Beta(당첨+1, 꽝+1)` 곡선을 만든다
2. 각 곡선에서 **숫자를 하나씩 뽑는다**
3. 뽑힌 숫자가 **가장 큰 기계**를 당긴다

`np.random.beta(a, b)`가 2번을 해줍니다.
곡선이 높은 곳이 잘 뽑히고, **곡선이 넓으면 넓은 범위에서 뽑힙니다.**

In [ ]:
def thompson(n_pulls=N):
    wins  = [0] * K       # 당첨 횟수
    loses = [0] * K       # 꽝 횟수
    total = 0
    history = []

    for t in range(n_pulls):
        # ✏️ 실습 ④ : 각 기계의 Beta(당첨+1, 꽝+1) 분포에서 숫자를 하나씩 뽑아 samples에 담으세요.
        #            (힌트: samples = [] 로 시작해서 for 반복으로 append 하세요)
        samples = 여기를_채우세요

        # ✏️ 실습 ⑤ : samples에서 가장 큰 값을 가진 기계의 번호를 i에 담으세요.
        #            (힌트: 빈칸 ②에서 쓴 argmax_random 을 그대로 쓰면 됩니다)
        i = 여기를_채우세요

        r = pull(i)
        if r == 1:
            wins[i]  += 1
        else:
            loses[i] += 1
        total += r
        history.append(total)

    trials = []
    for j in range(K):
        trials.append(wins[j] + loses[j])
    return total, trials, history


random.seed(0); np.random.seed(0)
score, trials, _ = thompson()
print(f'총점 : {score} / {N}   (최적 {BEST*N:.0f}점)')
for i in range(K):
    print(f'  기계 {NAMES[i]} (진짜 확률 {TRUE_P[i]:.2f}) : {trials[i]:4d}번 당김')

**시도 횟수를 보세요.**
가장 좋은 기계에 압도적으로 몰려 있고, 나쁜 기계도 **0번은 아닙니다.**
누가 시킨 게 아니라 분포에서 뽑다 보니 저절로 그렇게 된 겁니다.

## 6. 학습 과정 들여다보기

곡선이 실제로 어떻게 자라는지 스냅샷으로 확인해 봅시다.

In [ ]:
def thompson_trace(n_pulls=N, snaps=(0, 10, 50, 200, 1000)):
    wins, loses = [0]*K, [0]*K
    shots = {}
    if 0 in snaps:
        shots[0] = (wins[:], loses[:])

    for t in range(1, n_pulls + 1):
        samples = []
        for j in range(K):
            samples.append(np.random.beta(wins[j] + 1, loses[j] + 1))
        i = argmax_random(samples)      # np.argmax(samples) 를 써도 같습니다
        if pull(i) == 1: wins[i]  += 1
        else:            loses[i] += 1
        if t in snaps:
            shots[t] = (wins[:], loses[:])
    return shots


random.seed(1); np.random.seed(1)
snaps = (0, 10, 50, 200, 1000)
shots = thompson_trace(snaps=snaps)

colors = ['#0e8a52', '#0a7ec2', '#b06a00', '#6a3fc0', '#c8324a']
fig, axes = plt.subplots(1, len(snaps), figsize=(15, 2.9))

for ax, t in zip(axes, snaps):
    w, l = shots[t]
    top = 0
    for i in range(K):
        y = beta_dist.pdf(x, w[i] + 1, l[i] + 1)
        top = max(top, np.nanmax(y))
        ax.plot(x, y, color=colors[i], lw=2,
                label=NAMES[i] if t == snaps[-1] else None)
    ax.set_title(f'{t}번 당긴 뒤'); ax.set_xlabel('당첨 확률')
    ax.set_ylim(0, max(top * 1.15, 1.3))   # 칸마다 세로 배율이 다릅니다

axes[0].set_ylabel('밀도')
axes[-1].legend(title='기계', fontsize=8)
plt.suptitle('톰슨 샘플링이 배워가는 과정 — 곡선이 좁아지고 갈라진다', y=1.06)
plt.tight_layout(); plt.show()

w, l = shots[snaps[-1]]
print('마지막 상태')
for i in range(K):
    n_i = w[i] + l[i]
    est = w[i] / n_i if n_i else 0
    print(f'  {NAMES[i]} : Beta({w[i]+1:3d}, {l[i]+1:3d})  '
          f'시도 {n_i:3d}번  추정 {est:.2f}  (진짜 {TRUE_P[i]:.2f})')

**읽는 법**

- 맨 왼쪽(0번): 다섯 곡선이 **완전히 겹친** 평평한 직선 — 아무것도 모름
- 오른쪽으로 갈수록: 좋은 기계는 **오른쪽에서 좁고 높게**, 나쁜 기계는 왼쪽에 낮게
- 나쁜 기계의 곡선은 **여전히 넓습니다** — 몇 번 안 당겨봤으니까요.
  넓다는 건 언제든 큰 숫자를 뽑아 다시 기회를 얻을 수 있다는 뜻입니다.

## 7. 세 전략 정면 비교

In [ ]:
# N=1000 × 300회 반복 → 몇 초 걸립니다
results = {
    '그리디':          repeat(greedy,     runs=300),
    'ε-그리디 (0.1)':  repeat(eps_greedy, runs=300, eps=0.1),
    '톰슨 샘플링':      repeat(thompson,   runs=300),
}

print(f'{"전략":<16}{"평균":>8}{"표준편차":>10}{"최악":>7}{"최선":>7}')
print('-' * 50)
for name, sc in results.items():
    print(f'{name:<16}{sc.mean():8.1f}{sc.std():10.1f}{sc.min():7.0f}{sc.max():7.0f}')
print('-' * 50)
print(f'{"최적(천장)":<16}{BEST*N:8.1f}')
print(f'{"무작위(바닥)":<16}{CHANCE*N:8.1f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.4))

cols = {'그리디': '#c8324a', 'ε-그리디 (0.1)': '#b06a00', '톰슨 샘플링': '#6a3fc0'}
for name, sc in results.items():
    ax1.hist(sc, bins=30, alpha=.55, label=name, color=cols[name])
ax1.axvline(BEST * N, color='#0e8a52', ls='--', lw=2, label='최적')
ax1.set_title('점수 분포'); ax1.set_xlabel('점수'); ax1.legend(fontsize=8)

def avg_history(fn, runs=100, eps=None):
    '''여러 판의 누적 점수를 평균낸다 (한 판만 그리면 운에 휘둘립니다)'''
    acc = np.zeros(N)
    for run in range(runs):
        random.seed(run); np.random.seed(run)

        if eps is None:
            history = fn()[2]
        else:
            history = fn(eps=eps)[2]

        acc += np.array(history)
    return acc / runs

for name, fn, eps in [('그리디', greedy, None),
                      ('ε-그리디 (0.1)', eps_greedy, 0.1),
                      ('톰슨 샘플링', thompson, None)]:
    ax2.plot(avg_history(fn, eps=eps), lw=2, label=name, color=cols[name])
ax2.plot([0, N], [0, BEST * N], color='#0e8a52', ls='--', lw=2, label='최적')
ax2.set_title('누적 점수 (100판 평균)'); ax2.set_xlabel('당긴 횟수'); ax2.legend(fontsize=8)

plt.tight_layout(); plt.show()

**표준편차 열을 보세요.**
톰슨 샘플링은 평균이 높은 것보다도 **편차가 작은 게 더 중요합니다.**
운이 나빠도 크게 망하지 않는다 = 믿고 쓸 수 있다는 뜻입니다.

오른쪽 그래프에서 초록 점선(최적)에 가장 붙어 가는 선이 톰슨 샘플링입니다.
초반 200번쯤은 세 전략이 거의 붙어 있다가, 뒤로 갈수록 벌어지는 것도 눈여겨보세요.
톰슨 샘플링은 **초반에 여러 기계를 찔러보는 투자를 하고, 뒤에서 회수**합니다.

---

## 도전 과제

1. **`N`을 200으로 줄여보세요.** 톰슨 샘플링의 우위가 거의 사라집니다. 왜 그럴까요?
   (힌트: 톰슨 샘플링은 초반에 일부러 여러 기계를 찔러봅니다. 그 투자를 회수할 시간이 필요합니다.)

2. **기계를 더 어렵게 만들어보세요.**
   `TRUE_P = [0.51, 0.50, 0.49, 0.48, 0.47]`처럼 다섯 대가 거의 똑같으면 어떻게 되나요?

3. **ε을 시간에 따라 줄여보세요** (ε-decay).
   `eps = 1.0 / (t + 1)` 처럼 바꾸면 고정 ε보다 나아지나요?
   톰슨 샘플링은 이걸 손으로 안 해도 되는 이유를 설명해 보세요.

4. **내 문제로 바꿔보세요.**
   급식 메뉴 추천, 문제집 유형 고르기, 유튜브 썸네일 고르기 —
   `TRUE_P`가 무엇이고 `pull()`이 무엇인지만 정하면 그대로 돌아갑니다.

5. **어려움** — 중간에 확률이 바뀌면? (예: 100번째에 `TRUE_P`를 뒤섞기)
   톰슨 샘플링이 새 정답을 찾아낼까요? 얼마나 걸릴까요?